# Entrenamiento del Modelo — Detección de Enfermedades en Hojas de Café

Este notebook de **Google Colab** entrena, mediante *transfer learning* (MobileNetV2 + ImageNet),
un clasificador de imágenes capaz de distinguir entre las siguientes clases foliares del café:

- 🟠 **Roya** (*Hemileia vastatrix*)
- 🟣 **Phoma** / Cercospora (mancha foliar)
- 🟡 **Minador** (*Leucoptera coffeella*)
- 🟢 **Sano** (hoja sin síntomas)

**Salida del notebook:** `coffee_leaf_model.h5` + `class_names.json`, listos para copiarse a la
carpeta `model/` del repositorio del Servicio Web (Streamlit).

> Sugerencia: en Colab, activa GPU en `Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU`.

## 1. Preparar el entorno

In [ ]:
!pip install -q tensorflow pillow

import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("GPU disponible:", tf.config.list_physical_devices('GPU'))

## 2. Cargar el dataset

Sube el archivo `Dataset.zip` (el mismo que se entregó para el proyecto) a Colab, o móntalo desde
Google Drive. El zip contiene, entre otros, los siguientes archivos utilizables directamente:

- `coffee___rust.zip` → Roya
- `coffee__phoma.zip` → Phoma
- `coffee__leaf_miner.zip` → Minador
- `coffee___healthy.zip.001/.002/.003` (partes de un mismo zip) → Sano

> Nota: algunos archivos del dataset original (`cercospora_v2.0_fotoEstudio.zip`, `red_spider_v2.zip`,
> `Miner_Prueba.zip`, `Phoma_Prueba.zip`, `coffee___rust4.zip`) son **punteros Git LFS** (~130 bytes)
> y no contienen las imágenes reales; si tu equipo tiene acceso al repositorio Git LFS original,
> descárgalos primero con `git lfs pull` antes de subir el dataset a Colab para incluir más clases
> (p. ej. Cercospora y Ácaro/Araña roja) o más ejemplos por clase.

In [ ]:
from google.colab import files
uploaded = files.upload()  # selecciona Dataset.zip

In [ ]:
import zipfile, os, shutil, glob

RAW_DIR = "raw_dataset"
ORG_DIR = "dataset"  # dataset/<Clase>/*.jpg

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(ORG_DIR, exist_ok=True)

with zipfile.ZipFile("Dataset.zip", "r") as z:
    z.extractall(RAW_DIR)

base = os.path.join(RAW_DIR, "Dataset")

# Unir las partes del zip de hojas sanas (multi-volumen)
healthy_parts = sorted(glob.glob(os.path.join(base, "coffee___healthy.zip.*")))
if healthy_parts:
    with open(os.path.join(base, "coffee___healthy_full.zip"), "wb") as out:
        for part in healthy_parts:
            with open(part, "rb") as p:
                shutil.copyfileobj(p, out)

CLASS_ZIPS = {
    "Roya": "coffee___rust.zip",
    "Phoma": "coffee__phoma.zip",
    "Minador": "coffee__leaf_miner.zip",
    "Sano": "coffee___healthy_full.zip",
}

for class_name, zip_name in CLASS_ZIPS.items():
    zip_path = os.path.join(base, zip_name)
    if not os.path.exists(zip_path):
        print(f"[AVISO] No se encontró {zip_name}, se omite la clase {class_name}")
        continue
    extract_tmp = os.path.join(RAW_DIR, f"tmp_{class_name}")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_tmp)
    class_dir = os.path.join(ORG_DIR, class_name)
    os.makedirs(class_dir, exist_ok=True)
    for jpg in glob.glob(os.path.join(extract_tmp, "**", "*.jpg"), recursive=True):
        shutil.copy(jpg, class_dir)
    print(class_name, "->", len(os.listdir(class_dir)), "imágenes")

## 3. Crear los datasets de entrenamiento y validación

In [ ]:
IMG_SIZE = (160, 160)
BATCH_SIZE = 32
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    ORG_DIR, validation_split=0.2, subset="training", seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    ORG_DIR, validation_split=0.2, subset="validation", seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
)

class_names = train_ds.class_names
print("Clases:", class_names)

import json
with open("class_names.json", "w", encoding="utf-8") as f:
    json.dump(class_names, f, ensure_ascii=False, indent=2)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

## 4. Construir el modelo (Transfer Learning con MobileNetV2)

In [ ]:
from tensorflow.keras import layers, models

data_augmentation = models.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.1),
], name="augmentation")

base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet"
)
base_model.trainable = False  # fase 1: solo entrenamos la cabeza

preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(len(class_names), activation="softmax")(x)
model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

## 5. Fase 1: entrenar solo la cabeza clasificadora

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor="val_accuracy"),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3),
]

history = model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=callbacks)

## 6. Fase 2: *fine-tuning* (descongelar las últimas capas de MobileNetV2)

Esto suele mejorar la exactitud unos puntos porcentuales adicionales, a costa de más tiempo de
entrenamiento. Usa una tasa de aprendizaje baja para no destruir los pesos preentrenados.

In [ ]:
base_model.trainable = True
FINE_TUNE_AT = len(base_model.layers) - 30
for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

history_fine = model.fit(
    train_ds, validation_data=val_ds, epochs=10, callbacks=callbacks,
)

## 7. Evaluar y guardar el modelo final

In [ ]:
val_loss, val_acc = model.evaluate(val_ds)
print(f"Exactitud de validación final: {val_acc*100:.2f}%")

model.save("coffee_leaf_model.h5")
print("Modelo guardado como coffee_leaf_model.h5")

## 8. Matriz de confusión (opcional, recomendado para el informe)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

y_true, y_pred = [], []
for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap="Greens")
plt.title("Matriz de confusión - Validación")
plt.show()

## 9. Descargar los artefactos del modelo

Descarga `coffee_leaf_model.h5` y `class_names.json` y colócalos dentro de la carpeta
`model/` del repositorio del Servicio Web (Streamlit) antes de desplegar en Streamlit
Community Cloud.

In [ ]:
from google.colab import files
files.download("coffee_leaf_model.h5")
files.download("class_names.json")